# RL Project - Highway environment

In [8]:
import gymnasium as gym
import highway_env
import numpy as np
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
from shared_core_config import SHARED_CORE_CONFIG, SHARED_CORE_ENV_ID

In [9]:
env = gym.make(SHARED_CORE_ENV_ID, render_mode = None)
env.unwrapped.configure(SHARED_CORE_CONFIG)
env.reset()

(array([[ 1.        ,  0.8971738 ,  0.5       ,  0.3125    ,  0.        ],
        [ 1.        ,  0.09960218, -0.5       , -0.03671875,  0.        ],
        [ 1.        ,  0.1994654 ,  0.25      , -0.01355618,  0.        ],
        [ 1.        ,  0.29895815, -0.5       , -0.03685739,  0.        ],
        [ 1.        ,  0.39524472, -0.5       , -0.02413193,  0.        ],
        [ 1.        ,  0.4946119 ,  0.        , -0.029795  ,  0.        ],
        [ 1.        ,  0.60921395, -0.25      , -0.02046014,  0.        ],
        [ 1.        ,  0.71303374, -0.25      , -0.01899761,  0.        ],
        [ 1.        ,  0.8250566 ,  0.25      , -0.02656407,  0.        ],
        [ 1.        ,  0.9327459 , -0.5       , -0.04637612,  0.        ]],
       dtype=float32),
 {'speed': 25.0,
  'crashed': False,
  'action': np.int64(2),
  'rewards': {'collision_reward': 0.0,
   'right_lane_reward': 0.6666666666666666,
   'high_speed_reward': np.float64(0.375),
   'on_road_reward': 1.0}})

### Stable-baselines3 DQN

In [10]:
sb_DQN = DQN(policy="MlpPolicy",
             env=env,
             learning_rate=5e-4,
             buffer_size=50_000,
             learning_starts=1_000,
             batch_size=64,
             gamma=0.99,
             train_freq=4,
             target_update_interval=500,
             exploration_fraction=0.3,
             exploration_final_eps=0.05,
             verbose=1,
             tensorboard_log="./logs/sb3/"
             )

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


### DQN from scratch

In [11]:
import torch
import torch.nn as nn

class QNet(nn.Module):
    def __init__(self, obs_dim:int, n_actions:int, hidden:int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [12]:
from dataclasses import dataclass

@dataclass
class Batch:
    states: torch.Tensor
    actions: torch.Tensor
    rewards: torch.Tensor
    next_states: torch.Tensor
    dones: torch.Tensor

class ReplayBuffer:
    def __init__(self, capacity: int, obs_dim: int, device: str = "cpu"):
        self.capacity = capacity
        self.device = device
        self.ptr = 0
        self.size = 0

        self.states = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.dones = np.zeros(capacity, dtype=np.float32)

    def add(self, state, action, reward, next_state, done):
        self.states[self.ptr] = state
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.next_states[self.ptr] = next_state
        self.dones[self.ptr] = float(done)
        self.ptr  = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int) -> Batch:
        idx = np.random.randint(0, self.size, size=batch_size)
        return Batch(
            states      = torch.tensor(self.states[idx],      device=self.device),
            actions     = torch.tensor(self.actions[idx],     device=self.device),
            rewards     = torch.tensor(self.rewards[idx],     device=self.device),
            next_states = torch.tensor(self.next_states[idx], device=self.device),
            dones       = torch.tensor(self.dones[idx],       device=self.device),
        )

    def __len__(self):
        return self.size

In [13]:
import numpy as np
import torch
import torch.nn as nn
import copy

class DQNAgent:
    def __init__(
        self,
        obs_dim: int,
        n_actions: int,
        lr: float = 5e-4,
        gamma: float = 0.99,
        buffer_size: int = 50_000,
        batch_size: int = 64,
        eps_start: float = 1.0,
        eps_end: float = 0.05,
        eps_decay: int = 10_000,
        target_update_freq: int = 500,
        device: str = "cpu",
    ):
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.eps_start = eps_start
        self.eps_end = eps_end
        self.eps_decay = eps_decay
        self.target_update_freq = target_update_freq
        self.device = device
        self.steps = 0

        self.q_net      = QNet(obs_dim, n_actions).to(device)
        self.target_net = copy.deepcopy(self.q_net).to(device)
        self.target_net.eval()

        self.optimizer = torch.optim.Adam(self.q_net.parameters(), lr=lr)
        self.loss_fn   = nn.SmoothL1Loss()  # Huber loss
        self.buffer    = ReplayBuffer(buffer_size, obs_dim, device)

    @property
    def epsilon(self) -> float:
        return self.eps_end + (self.eps_start - self.eps_end) * \
               np.exp(-self.steps / self.eps_decay)

    def select_action(self, state: np.ndarray) -> int:
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        state_t = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            return self.q_net(state_t).argmax(dim=1).item()

    def store(self, state, action, reward, next_state, done):
        self.buffer.add(state, action, reward, next_state, done)

    def train_step(self) -> float | None:
        if len(self.buffer) < self.batch_size:
            return None

        batch = self.buffer.sample(self.batch_size)

        # Q(s, a)
        q_values = self.q_net(batch.states).gather(1, batch.actions.unsqueeze(1)).squeeze(1)

        # TD target: r + γ · max_a' Q_target(s', a')
        with torch.no_grad():
            next_q = self.target_net(batch.next_states).max(dim=1).values
            td_target = batch.rewards + self.gamma * next_q * (1 - batch.dones)

        loss = self.loss_fn(q_values, td_target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), 10.0)
        self.optimizer.step()

        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())

        return loss.item()

In [14]:

env = gym.make("highway-v0")
env.unwrapped.configure(SHARED_CORE_CONFIG)
obs, _ = env.reset()
obs = obs.flatten()
obs_dim   = obs.shape[0]
n_actions = env.action_space.n

agent = DQNAgent(obs_dim=obs_dim, n_actions=n_actions, device="cpu")

n_episodes = 500
for ep in range(n_episodes):
    state, _ = env.reset()
    state = state.flatten()
    total_reward = 0.0

    while True:
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        next_state = next_state.flatten()
        done = terminated or truncated

        agent.store(state, action, reward, next_state, done)
        loss = agent.train_step()
        state = next_state
        total_reward += reward

        if done:
            break

    if ep % 20 == 0:
        print(f"Ep {ep:4d} | reward={total_reward:.2f} | eps={agent.epsilon:.3f}")

Ep    0 | reward=11.13 | eps=1.000


KeyboardInterrupt: 